In [0]:
import pyspark.sql.functions as F

df_bronze = spark.read.table("bronze_products")

try:
    df_silver = df_bronze.select(
        F.col("id").alias("product_id"),
        F.col("title").alias("product_name"),
        F.col("category"),
        F.col("price"),
        F.col("discountPercentage"),
        F.col("rating"),
        F.col("stock").cast("int"),
        F.col("warrantyInformation")
    )

    df_silver = df_silver.filter(F.col("stock") > 0)

    df_silver = df_silver.withColumn("final_usd_price", F.round(F.col("price") * (1 - F.col("discountPercentage") / 100), 2))
    
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_products")

    print("Transformed data written to silver_products")
except Exception as e:
    print(e)
